# State Objective Fold-Local BC Readiness v1

Goal: convert the previous BC bridge into a safer next-step training plan. The prior notebook found that direct `balanced_rule_residual_v22` hybrid-delta labels are degenerate. This notebook checks whether the healthier `risk_budget_usage_v1` label surface is trainable only after the state-objective targets are regenerated inside each walk-forward train fold.

External basis used for this decision:
- Qlib keeps the research workflow reproducible and artifact-driven: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL's train/validation/test discipline supports keeping training targets separate from held-out evaluation: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Behavior cloning / offline RL can stabilize policy initialization, but only when demonstrations are in the allowed training scope: https://doi.org/10.21314/JCF.2016.322

This is a no-promotion readiness audit. It does not train or promote a model.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.soxl_quant.app.state_objective_training_targets import (
    filter_state_objective_targets_for_train_window,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
)

CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_fold_local_bc_readiness_v1"
SOURCE_CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_bc_warm_start_v1"
TARGETS_PATH = ROOT / "runtime" / "model_selection" / "card" / "state_objective_training_target_v1" / "training_targets.csv"
CARD.mkdir(parents=True, exist_ok=True)

profile_labels = pd.read_csv(SOURCE_CARD / "profile_label_comparison.csv")
targets = pd.read_csv(TARGETS_PATH, parse_dates=["datetime"])
profile_labels

,profile,obs_dim,samples,unique_actions,dominant_action,dominant_action_share,action_entropy
0,risk_budget_usage_v1,315,755,5,0,0.422517,1.300036
1,balanced,79,755,10,0,0.911258,0.442636
2,balanced_rule_residual_v22,340,755,5,0,0.970861,0.155705


## Current Target Scope Gate

The previous label distribution says `risk_budget_usage_v1` is the best action surface. This gate asks a different question: are the current targets safe to use for training? Answer: no, because they are global OOS / walk-forward target rows.

In [2]:
current_scope_scores = score_state_objective_bc_readiness(
    profile_labels,
    target_scope="global_oos_walkforward",
)
current_scope_summary = summarize_state_objective_bc_readiness(current_scope_scores)
current_scope_scores.to_csv(CARD / "current_scope_readiness_scores.csv", index=False)
current_scope_summary

{'decision': 'state_objective_bc_scope_blocked_no_promotion',
 'selected_profile': 'risk_budget_usage_v1',
 'selected_target_scope': 'global_oos_walkforward',
 'selected_bc_readiness_gate': False,
 'selected_label_distribution_gate': True,
 'selected_fold_local_scope_gate': False,
 'selected_dominant_action_share': 0.4225165562913907,
 'selected_unique_actions': 5,
 'selected_samples': 755,
 'selected_bc_readiness_loss': 100.0,
 'next_mechanism': 'generate_fold_local_state_objective_targets'}

## Fold-Local Counterfactual Gate

This uses the same label-distribution evidence but flips only the target scope to `fold_local`. It shows which action surface should be used once targets are regenerated within each walk-forward train fold.

In [3]:
fold_local_scores = score_state_objective_bc_readiness(
    profile_labels,
    target_scope="fold_local",
)
fold_local_summary = summarize_state_objective_bc_readiness(fold_local_scores)
fold_local_scores.to_csv(CARD / "fold_local_readiness_scores.csv", index=False)
fold_local_summary

{'decision': 'state_objective_bc_ready_for_training',
 'selected_profile': 'risk_budget_usage_v1',
 'selected_target_scope': 'fold_local',
 'selected_bc_readiness_gate': True,
 'selected_label_distribution_gate': True,
 'selected_fold_local_scope_gate': True,
 'selected_dominant_action_share': 0.4225165562913907,
 'selected_unique_actions': 5,
 'selected_samples': 755,
 'selected_bc_readiness_loss': 0.0,
 'next_mechanism': 'run_fold_local_bc_warm_start_then_walkforward_ppo'}

## Walk-Forward Fold Plan

The helper below filters target rows to non-overlapping train windows and rejects any train/eval overlap. These rows are still marked audit-only because the current target file was built from global OOS/walk-forward evidence. The next training run must regenerate the target file inside each train fold before BC warm-start.

In [4]:
folds = [
    {"fold": "wf_2022", "train_start": "2020-01-02", "train_end": "2021-12-31", "eval_start": "2022-01-03", "eval_end": "2022-12-30"},
    {"fold": "wf_2023_2024", "train_start": "2020-01-02", "train_end": "2022-12-30", "eval_start": "2023-01-03", "eval_end": "2024-12-31"},
]
fold_rows = []
for fold in folds:
    train_targets = filter_state_objective_targets_for_train_window(
        targets,
        train_start=fold["train_start"],
        train_end=fold["train_end"],
        eval_start=fold["eval_start"],
        eval_end=fold["eval_end"],
    )
    fold_rows.append(
        {
            **fold,
            "audit_target_rows": int(len(train_targets)),
            "audit_unique_dates": int(train_targets["datetime"].dt.normalize().nunique()),
            "audit_rebound_rows": int(train_targets["is_rebound_cluster"].sum()) if "is_rebound_cluster" in train_targets else 0,
            "audit_downtrend_rows": int(train_targets["is_downtrend_state"].sum()) if "is_downtrend_state" in train_targets else 0,
            "target_scope": "global_audit_only",
            "training_safe": False,
            "required_next_step": "regenerate targets using only rows <= train_end before BC warm-start",
        }
    )
fold_plan = pd.DataFrame(fold_rows)
fold_plan.to_csv(CARD / "fold_local_bc_fold_plan.csv", index=False)
fold_plan

,fold,train_start,train_end,eval_start,eval_end,audit_target_rows,audit_unique_dates,audit_rebound_rows,audit_downtrend_rows,target_scope,training_safe,required_next_step
0,wf_2022,2020-01-02,2021-12-31,2022-01-03,2022-12-30,757,505,0,352,global_audit_only,False,regenerate targets using only rows <= train_en...
1,wf_2023_2024,2020-01-02,2022-12-30,2023-01-03,2024-12-31,1258,756,0,838,global_audit_only,False,regenerate targets using only rows <= train_en...


## Decision

The next runnable optimization should not train v22 hybrid-delta labels. It should regenerate state-objective targets inside each walk-forward training fold, then build `risk_budget_usage_v1` BC warm-start labels, then run PPO/checkpoint retention against the +30pp excess and <=15% MaxDD gates.

In [5]:
selection = {
    "experiment": "state_objective_fold_local_bc_readiness_v1",
    "decision": "fold_local_bc_plan_ready_current_targets_scope_blocked",
    "promotion_allowed": False,
    "selected_profile_for_next_training": fold_local_summary["selected_profile"],
    "current_scope_decision": current_scope_summary["decision"],
    "fold_local_counterfactual_decision": fold_local_summary["decision"],
    "next_mechanism": "regenerate_fold_local_targets_then_budget_usage_bc_warm_start_ppo",
    "training_scope_requirement": "no target row may be generated from eval/OOS dates for that fold",
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": 30.0,
        "max_dd_max_pct": 15.0,
        "walkforward_required": True,
        "label_distribution_gate": True,
        "fold_local_scope_gate": True,
    },
    "sidecars": [
        "current_scope_readiness_scores.csv",
        "fold_local_readiness_scores.csv",
        "fold_local_bc_fold_plan.csv",
        "summary.json",
    ],
}
summary = {
    "current_scope_summary": current_scope_summary,
    "fold_local_summary": fold_local_summary,
    "fold_plan_rows": fold_plan.to_dict(orient="records"),
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True))
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True))
selection

{'experiment': 'state_objective_fold_local_bc_readiness_v1',
 'decision': 'fold_local_bc_plan_ready_current_targets_scope_blocked',
 'promotion_allowed': False,
 'selected_profile_for_next_training': 'risk_budget_usage_v1',
 'current_scope_decision': 'state_objective_bc_scope_blocked_no_promotion',
 'fold_local_counterfactual_decision': 'state_objective_bc_ready_for_training',
 'next_mechanism': 'regenerate_fold_local_targets_then_budget_usage_bc_warm_start_ppo',
 'training_scope_requirement': 'no target row may be generated from eval/OOS dates for that fold',
 'acceptance_gate_for_next_run': {'relative_return_min_pp': 30.0,
  'max_dd_max_pct': 15.0,
  'walkforward_required': True,
  'label_distribution_gate': True,
  'fold_local_scope_gate': True},
 'sidecars': ['current_scope_readiness_scores.csv',
  'fold_local_readiness_scores.csv',
  'fold_local_bc_fold_plan.csv',
  'summary.json']}